# ระบบแจ้งเตือนเครื่องซักผ้าร้านหยอดเหรียญ

หัวข้อ mini project "ระบบแจ้งเตือนเครื่องซักผ้า"
Python Data Structures (Dictionary, Set, Frozenset)

ผู้ใช้งาน 2 กลุ่ม:
1. คนที่ต้องการใช้บริการร้านซักผ้าหยอดเหรียญทียังไม่ไปร้าน -> อยากรู้ว่ามีเครื่องว่างกี่เครื่อง
2. คนที่ได้ใช้บริการซักผ้าอยู่แล้ว -> อยากรู้ว่าต้องไปรับผ้าตอนไหน


## Frozenset สำหรับค่าคงที่ของระบบ (Immutable Config)

LO1: Frozenset สำหรับค่าคงที่ของระบบ (Immutable Config)
สถานะที่ระบบยอมรับเท่านั้น ห้ามแก้ไขระหว่างรันโปรแกรม

In [1]:
VALID_STATUSES = frozenset({
    "ว่าง",
    "เครื่องกำลังซัก",
    "เครื่องกำลังปั่นผ้า",
    "พร้อมรับผ้าในอีก 5 นาที",
    "พร้อมรับผ้า",
})

Lab4 (Tuple): บัญชีผู้ดูแลร้าน - รองรับแอดมินเพียง 1 คน
ใช้ tuple เพราะเป็นข้อมูลคงที่ ไม่มีการเพิ่ม/ลบสมาชิกระหว่างรันโปรแกรม

โครงสร้าง: (username, password)

In [2]:
ADMIN_ACCOUNT = ("ผู้ดูแลร้าน", "shop1234")

MAX_LOGIN_ATTEMPTS = 3  # Lab3 (Loop): จำกัดจำนวนครั้งที่กรอกรหัสผิด

## Dictionary สถานะเครื่องซักผ้าแต่ละเครื่อง

 LO1: Dictionary สำหรับสถานะของเครื่องแต่ละเครื่อง (Machine ID -> Detail)

In [3]:
machines = {
    1: {"status": "ว่าง", "remaining_min": 0},
    2: {"status": "เครื่องกำลังซัก", "remaining_min": 30},
    3: {"status": "เครื่องกำลังปั่นผ้า", "remaining_min": 10},
    4: {"status": "ว่าง", "remaining_min": 0},
    5: {"status": "พร้อมรับผ้าในอีก 5 นาที", "remaining_min": 5},
    6: {"status": "พร้อมรับผ้า", "remaining_min": 0},
    7: {"status": "ว่าง", "remaining_min": 0},
    8: {"status": "เครื่องกำลังซัก", "remaining_min": 30},
    9: {"status": "ว่าง", "remaining_min": 0},
    10: {"status": "เครื่องกำลังปั่นผ้า", "remaining_min": 10},
}

## Set: หาเครื่องที่ว่างตอนนี้

LO2: Set สำหรับหาเครื่องที่ว่างตอนนี้ (คำนวณจาก dictionary)

In [4]:
def get_available_machines(machines_dict):
    """คืนค่าเป็น set ของหมายเลขเครื่องที่สถานะเป็น 'ว่าง'"""
    return {mid for mid, info in machines_dict.items() if info["status"] == "ว่าง"}

## Feature

กลุ่ม A: เช็คเครื่องว่างก่อนไปร้าน

In [5]:
def check_available(machines_dict):
    available = get_available_machines(machines_dict)  # Set
    print(f"\nเครื่องว่างตอนนี้: {len(available)} เครื่อง")
    if available:
        print(f"หมายเลข: {sorted(available)}")
    else:
        print("ไม่มีเครื่องว่าง กรุณารอสักครู่")

กลุ่ม B: เช็คเครื่องของตัวเอง (ต้องไปรับผ้าตอนไหน)

In [6]:
def check_my_machine(machines_dict, machine_id):
    # LO4: Defensive check ก่อนเข้าถึง dictionary key
    if machine_id not in machines_dict:
        print("ขออภัย ไม่พบเครื่องหมายเลขนี้")
        return

    info = machines_dict[machine_id]
    status = info["status"]
    remaining = info["remaining_min"]

    if status == "พร้อมรับผ้า":
        print(f"เครื่องหมายเลข {machine_id}: พร้อมรับผ้าของคุณแล้ว! ไปรับที่ร้านได้เลย")
    elif status == "พร้อมรับผ้าในอีก 5 นาที":
        print(f"เครื่องหมายเลข {machine_id}: เหลืออีก ~{remaining} นาที รับผ้าที่ร้านได้!")
    elif status == "ว่าง":
        print(f"เครื่องหมายเลข {machine_id}: ยังว่างอยู่ สามารถนำผ้ามาซักได้!")
    else:
        print(f"เครื่องหมายเลข {machine_id}: สถานะ '{status}' เหลืออีก ~{remaining} นาที")

ดูสถานะทุกเครื่อง

In [7]:
def show_all_status(machines_dict):
    print("\n--- สถานะเครื่องซักผ้าทั้งหมด ---")
    for mid, info in machines_dict.items():
        print(f"เครื่องหมายเลข {mid:>2}: {info['status']:<15} (~{info['remaining_min']} นาที)")

Defensive Programming: อัปเดตสถานะเครื่อง (สำหรับแอดมิน/เซนเซอร์)

ตรวจสอบกับ frozenset ก่อนทุกครั้งว่าสถานะใหม่ถูกต้องหรือไม่

In [8]:
def update_status(machines_dict, machine_id, new_status, remaining_min=0):
    if new_status not in VALID_STATUSES:
        print(f"สถานะ '{new_status}' ไม่ถูกต้อง ระบบไม่รองรับ")
        print(f"   สถานะที่รองรับ: {tuple(VALID_STATUSES)}")
        return False

    if machine_id not in machines_dict:
        print(f"ขออภัย ไม่พบเครื่องหมายเลข {machine_id}")
        return False

    machines_dict[machine_id]["status"] = new_status
    machines_dict[machine_id]["remaining_min"] = remaining_min
    print(f"อัปเดตเครื่อง {machine_id} เป็น '{new_status}' แล้ว")
    return True

Admin Authentication - รวมความรู้ Lab1-4:

Lab1: input/print พื้นฐาน

Lab2: if/elif conditionals

Lab3: while loop จำกัดจำนวนครั้ง

Lab4: tuple เก็บบัญชีคงที่ 1 ชุด

In [9]:
def admin_login():
    """คืนค่า True ถ้า login สำเร็จ, False ถ้ากรอกผิดเกินจำนวนครั้งที่กำหนด"""
    admin_username, admin_password = ADMIN_ACCOUNT  # Lab4: unpack tuple
    attempts = 0  # Lab1: ตัวแปรพื้นฐาน

    while attempts < MAX_LOGIN_ATTEMPTS:  # Lab3: control flow loop
        username = input("Username ผู้ดูแลร้าน: ").strip()
        password = input("Password: ").strip()

        # Lab2: เทียบค่ากับบัญชีแอดมินที่มีอยู่เพียงชุดเดียว
        if username == admin_username and password == admin_password:
            print(f"เข้าสู่ระบบสำเร็จ ยินดีต้อนรับ {username}")
            return True
        else:
            attempts += 1
            remaining = MAX_LOGIN_ATTEMPTS - attempts
            if remaining > 0:
                print(f"Username หรือ Password ไม่ถูกต้อง (เหลือโอกาส {remaining} ครั้ง)")

    print("กรอกผิดเกินกำหนด ปฏิเสธการเข้าถึงเมนูผู้ดูแลร้าน")
    return False

## Interactive Menu Loop (Main Program)

LO3: Interactive Menu Loop

In [10]:
def main():
    while True:
        print("\n=== ระบบแจ้งเตือนเครื่องซักผ้าร้านหยอดเหรียญ ===")
        print("1. เช็คเครื่องว่าง (ก่อนไปที่ร้าน)")
        print("2. เช็คเครื่องของฉัน (กำลังซักอยู่)")
        print("3. ดูสถานะทุกเครื่อง")
        print("4. [ผู้ดูแลร้าน] อัปเดตสถานะเครื่อง")
        print("5. ออกจากโปรแกรม")

        choice = input("\nเลือกเมนู (1-5): ").strip()

        if choice == "1":
            check_available(machines)

        elif choice == "2":
            try:
                mid = int(input("เครื่องหมายเลข: ").strip())
                check_my_machine(machines, mid)
            except ValueError:
                print("กรุณาใส่ตัวเลขเท่านั้น")

        elif choice == "3":
            show_all_status(machines)

        elif choice == "4":
            print("\n--- ต้องเข้าสู่ระบบผู้ดูแลร้านก่อน ---")
            if admin_login():
                try:
                    mid = int(input("เครื่องหมายเลข: ").strip())
                    print(f"สถานะที่รองรับ: {tuple(VALID_STATUSES)}")
                    status = input("สถานะใหม่: ").strip()
                    mins = int(input("เวลาที่เหลือ (นาที): ").strip() or 0)
                    update_status(machines, mid, status, mins)
                except ValueError:
                    print("**กรุณาใส่ตัวเลขเท่านั้น**")
            # ถ้า login ไม่สำเร็จ จะกลับไปที่เมนูหลักโดยอัตโนมัติ

        elif choice == "5":
            print("ขอบคุณที่ใช้บริการ!")
            break

        else:
            print("กรุณาเลือก 1-5")

## เรียกใช้งานโปรแกรม

In [12]:
if __name__ == "__main__":
    main()


=== ระบบแจ้งเตือนเครื่องซักผ้าร้านหยอดเหรียญ ===
1. เช็คเครื่องว่าง (ก่อนไปที่ร้าน)
2. เช็คเครื่องของฉัน (กำลังซักอยู่)
3. ดูสถานะทุกเครื่อง
4. [ผู้ดูแลร้าน] อัปเดตสถานะเครื่อง
5. ออกจากโปรแกรม

เลือกเมนู (1-5): 1

เครื่องว่างตอนนี้: 4 เครื่อง
หมายเลข: [1, 4, 7, 9]

=== ระบบแจ้งเตือนเครื่องซักผ้าร้านหยอดเหรียญ ===
1. เช็คเครื่องว่าง (ก่อนไปที่ร้าน)
2. เช็คเครื่องของฉัน (กำลังซักอยู่)
3. ดูสถานะทุกเครื่อง
4. [ผู้ดูแลร้าน] อัปเดตสถานะเครื่อง
5. ออกจากโปรแกรม

เลือกเมนู (1-5): 2
เครื่องหมายเลข: 3
เครื่องหมายเลข 3: สถานะ 'เครื่องกำลังปั่นผ้า' เหลืออีก ~10 นาที

=== ระบบแจ้งเตือนเครื่องซักผ้าร้านหยอดเหรียญ ===
1. เช็คเครื่องว่าง (ก่อนไปที่ร้าน)
2. เช็คเครื่องของฉัน (กำลังซักอยู่)
3. ดูสถานะทุกเครื่อง
4. [ผู้ดูแลร้าน] อัปเดตสถานะเครื่อง
5. ออกจากโปรแกรม

เลือกเมนู (1-5): 3

--- สถานะเครื่องซักผ้าทั้งหมด ---
เครื่องหมายเลข  1: ว่าง            (~0 นาที)
เครื่องหมายเลข  2: เครื่องกำลังซัก (~30 นาที)
เครื่องหมายเลข  3: เครื่องกำลังปั่นผ้า (~10 นาที)
เครื่องหมายเลข  4: ว่าง            (~0 นาที)
